In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

## Load Experiment

First, we load the experiment from the `results` folder. Each experiment is collected into a single `.parquet` file. Its rows contain recognition results for each configuration (combination) and each fold (flattened).

In [2]:
# Configuration which experiment should be loaded.
EXPERIMENT_NAMES = [
    "sface_anonymizers_no_aggregation",
    "arcface_anonymizers_no_aggregation",
    "ghostfacenet_anonymizers_no_aggregation",
]
RESULTS_PATH = Path("../../results")

In [3]:
# Load the experiment dataset.
EXPERIMENT_DATA = pd.concat([pd.read_parquet(RESULTS_PATH / f"{name}.parquet") for name in EXPERIMENT_NAMES])
EXPERIMENT_DATA

,identity,video,observations,top-1,top-5,top-10,predicted,fold,gallery_frame_anonymization,gallery_frame_aggregation,gallery_frame_embedding,gallery_embedding_aggregation,samples_frame_anonymization,samples_frame_aggregation,samples_frame_embedding,samples_embedding_aggregation,score_aggregation,pipeline
0,Aaron_Eckhart,2,56,1.0,1.0,1.0,Aaron_Eckhart,1,none,video_none,sface,video_none,none,video_none,sface,video_none,max_max,frame
1,Abdel_Aziz_Al-Hakim,0,75,1.0,1.0,1.0,Abdel_Aziz_Al-Hakim,1,none,video_none,sface,video_none,none,video_none,sface,video_none,max_max,frame
2,Abdullah_Gul,1,61,1.0,1.0,1.0,Abdullah_Gul,1,none,video_none,sface,video_none,none,video_none,sface,video_none,max_max,frame
3,Abel_Pacheco,3,52,1.0,1.0,1.0,Abel_Pacheco,1,none,video_none,sface,video_none,none,video_none,sface,video_none,max_max,frame
4,Adam_Sandler,0,48,1.0,1.0,1.0,Adam_Sandler,1,none,video_none,sface,video_none,none,video_none,sface,video_none,max_max,frame
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60539,Yuri_Fedotov,4,1,0.0,0.0,1.0,Jack_Valenti,3,pixelation,video_none,ghostfacenet,video_none,pixelation,video_facial,ghostfacenet,video_none,max_max,frame
60540,Zhang_Ziyi,1,1,0.0,0.0,0.0,Kit_Bond,3,pixelation,video_none,ghostfacenet,video_none,pixelation,video_facial,ghostfacenet,video_none,max_max,frame
60541,Zhong_Nanshan,5,1,1.0,1.0,1.0,Zhong_Nanshan,3,pixelation,video_none,ghostfacenet,video_none,pixelation,video_facial,ghostfacenet,video_none,max_max,frame
60542,Zoran_Djindjic,2,1,0.0,1.0,1.0,Leticia_Van_de_Putte,3,pixelation,video_none,ghostfacenet,video_none,pixelation,video_facial,ghostfacenet,video_none,max_max,frame


## Visualize accuracy

The following compares the recognition accuracy (i.e., top-1, top-5, top-10) of the three recognizers
against each sample anonymization, with no aggregation on either leg.

Each **row** is one gallery anonymization — what the attacker holds. The top row (`Gallery: None`) is the
headline pairing: a clear gallery queried with anonymized samples. The rows below it ask whether an
attacker whose own gallery is anonymized does better by matching like against like; those cells sit where
the row's anonymization meets the matching column.

Keying on the gallery condition rather than pooling it matters: averaging all four rows into one panel
roughly halves the clear/clear cell (ArcFace reads 0.49 pooled against 0.96 on a clear gallery). An
assertion below fails if any *other* unkeyed configuration column varies.

In [4]:
# Rename labels once, on the full dataset. Both legs are renamed for both anonymization and
# recognizer, so the legs' values are directly comparable -- the pairing check below, and the
# shared ordering that puts like-against-like on the diagonal.
RECOGNIZER_LABELS = {"sface": "SFace", "arcface": "ArcFace", "ghostfacenet": "GhostFaceNet"}
ANONYMIZATION_LABELS = {
    "none": "None",
    "gaussianblur": "Gaussian blur",
    "blackbar": "Black bar",
    "pixelation": "Pixelation",
}
data = EXPERIMENT_DATA.replace(
    {
        "samples_frame_anonymization": ANONYMIZATION_LABELS,
        "gallery_frame_anonymization": ANONYMIZATION_LABELS,
        "gallery_frame_embedding": RECOGNIZER_LABELS,
        "samples_frame_embedding": RECOGNIZER_LABELS,
    }
)

# One column per metric, one row per gallery anonymization -- what the attacker holds.
# The top row is the headline pairing (clear gallery, anonymized samples); the rows below ask
# whether an attacker whose own gallery is anonymized does better matching like against like.
metrics = ["top-1", "top-5", "top-10"]

# Shared ordering across all panels (based on one reference metric). Both legs carry the same
# four anonymizations, so they share an order and the like-against-like cells line up.
ORDER_METRIC = "top-1"
agg_order = data.groupby("gallery_frame_embedding")[ORDER_METRIC].mean().sort_values(ascending=False).index
anonymization_order = (
    data.groupby("samples_frame_anonymization")[ORDER_METRIC].mean().sort_values(ascending=False).index
)
gallery_anonymizations = [a for a in anonymization_order if a in set(data["gallery_frame_anonymization"])]

# Guard against pooling a dimension: every configuration column the plot does not key on must
# be constant, or its cells average over conditions the figure never names. The two legs'
# recognizers are checked as a pair, since the index names one recognizer for both.
KEYED = {
    "gallery_frame_embedding",
    "samples_frame_embedding",
    "samples_frame_anonymization",
    "gallery_frame_anonymization",
}
config_columns = [c for c in data.columns if c.startswith(("gallery_", "samples_"))] + ["score_aggregation", "pipeline"]
assert (data["gallery_frame_embedding"] == data["samples_frame_embedding"]).all(), (
    "the legs use different recognizers, so the index no longer names one recognizer"
)
pooled = {c: sorted(data[c].unique()) for c in config_columns if c not in KEYED and data[c].nunique() > 1}
assert not pooled, f"unkeyed columns vary, so each cell pools them: {pooled}"

# Build all pivots first so we can find a shared color range
pivots = {
    (m, g): data[data["gallery_frame_anonymization"] == g]
    .pivot_table(
        index="gallery_frame_embedding",
        columns="samples_frame_anonymization",
        values=m,
        aggfunc="mean",
    )
    .reindex(index=agg_order, columns=anonymization_order)
    for m in metrics
    for g in gallery_anonymizations
}
vmin = min(p.min().min() for p in pivots.values())
vmax = max(p.max().max() for p in pivots.values())

# Spread per cell: "std" for standard deviation, "sem" for standard error of the mean
SPREAD = "std"

spreads = {
    (m, g): data[data["gallery_frame_anonymization"] == g]
    .pivot_table(
        index="gallery_frame_embedding",
        columns="samples_frame_anonymization",
        values=m,
        aggfunc=SPREAD,
    )
    .reindex(index=agg_order, columns=anonymization_order)
    for m in metrics
    for g in gallery_anonymizations
}

def make_labels(mean_df, spread_df):
    return (
            mean_df.map(lambda v: f"{v:.2f}")
            + "\n± "
            + spread_df.map(lambda v: f"{v:.2f}" if pd.notna(v) else "n/a")
    )

fig, axes = plt.subplots(
    len(gallery_anonymizations),
    len(metrics),
    figsize=(6 * len(metrics), 4.5 * len(gallery_anonymizations)),
    sharex=True,
    sharey=True,
    squeeze=False,  # always 2D, even with a single row or column
)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  # one colorbar for all panels

for r, g in enumerate(gallery_anonymizations):
    for c, m in enumerate(metrics):
        ax = axes[r, c]
        first = r == 0 and c == 0
        sns.heatmap(
            pivots[(m, g)],
            ax=ax,
            annot=make_labels(pivots[(m, g)], spreads[(m, g)]),
            fmt="",
            annot_kws={"fontsize": 9},
            #annot=True,
            #fmt=".3f",
            cmap="BuGn",
            vmin=vmin,
            vmax=vmax,
            linewidths=0.5,
            linecolor="w",
            cbar=first,
            cbar_ax=cbar_ax if first else None,
            cbar_kws={"label": "Mean Accuracy"},
        )
        # Column titles (metric) on top row only
        ax.set_title(m if r == 0 else "")
        # x-label on bottom row only
        ax.set_xlabel("Anonymization (samples)" if r == len(gallery_anonymizations) - 1 else "")
        # Row label (gallery anonymization) on first column only
        ax.set_ylabel(f"Gallery: {g}\nRecognizer (gallery)" if c == 0 else "")

fig.suptitle("Recognition accuracy by gallery anonymization, recognizer and sample anonymization", y=0.97)
fig.subplots_adjust(right=0.9, top=0.93, wspace=0.1, hspace=0.15)
plt.show()

AssertionError: unkeyed columns vary, so each cell pools them: {'samples_frame_aggregation': ['video_facial', 'video_none']}